## 0 · One Kick, One Question
metadata
language
> **Match day. 20 metres from goal. One free kick.**
>
> The ball draws one smooth curve, but a simulation does not need to know the whole curve in advance. It can build the flight from many tiny questions:
>
> 1. Where is the ball now?
> 2. How far does horizontal velocity carry it during a tiny time slice?
> 3. How far does vertical velocity carry it during that same slice?
> 4. How much does gravity change vertical velocity for the next slice?
> 5. Repeat.
>
> That is calculus in action: use change **at this instant** to predict the **next small step**, then let many small steps accumulate into a journey.

The model uses 28 equal time slices. A cyan bar records the next horizontal change, a gold bar records the next vertical change, and gravity prepares the vertical velocity used by the following step.

Start with the pattern: the vertical changes begin positive, shrink near the top, pass through zero, and become negative as the ball falls.

![Gravity-only free-kick trajectory assembled from repeated local updates, clearing the wall and reaching the target](images/gravity-only-trajectory.svg)

**Read the construction:**

- the cyan bar asks, “How far forward during this tiny moment?”
- the gold bar asks, “Will the next point be higher or lower?”
- each sampled point lies on the same gravity-only trajectory;
- the path clears the wall, stays inside the goal, and reaches the chosen target.

No single update knows the whole curve. The journey appears because the same local question is asked again and again.

# Mathematical Foundations for Machine Learning

## Follow the Match in Time

The mathematics will arrive in the same order as the kick.

### 1. At contact: set the initial velocity

The player sees the ball, wall, goal, and target. At contact, the foot gives the ball an initial direction and speed. An **arrow** records both: its direction says where the ball starts moving, and its length records the speed. Mathematics calls that arrow a **vector**.

Two arrows help us describe the shot:

- the **target vector** points directly from the ball to the intended finish point;
- the **kick vector** is the ball's initial velocity as it leaves the foot.

Without gravity, the kick could point directly along the target vector. In this model, gravity continuously pulls downward during the flight, so the kick must start above the direct target line. The player sets the initial velocity; gravity turns that initial condition into the curved path and final position.

We will first reason from rise, run, travel time, and gravity to the needed launch direction. Only then will we name the trigonometric relationships and compare the two arrows with a **dot product**.

### 2. After contact: predict the next instant

Once the ball is moving, its current velocity predicts one nearby point. This is the local-change idea behind a **derivative**.

### 3. During the flight: repeat

One nearby prediction is not a journey. Repeating and adding the tiny changes builds the full arc. This is the accumulation idea behind **integration**.

### 4. Across many kicks: allow variation

One exact kick follows one curve. Repeated kicks form a spread of possible outcomes, which introduces **probability**.

> **First see the physical question. Then give its mathematical tool a name.**

## Match Geometry Used by the Simulation

The free kick must satisfy visible constraints:

- launch speed: **20 m/s**;
- wall: **9.15 m** away and **1.8 m** high;
- goal line: **20 m** away with a **2.44 m** crossbar;
- back-net target: about **21.5 m** away and **1.10 m** high.

The ball has radius `0.11 m`, so its centre must clear the wall by more than the wall height alone and stay fully inside the goal frame.

> **Simplifying assumption:** after contact, gravity is the only force in the model. We deliberately ignore air resistance, wind, and spin effects such as topspin, dip, and curl.

The next code cell only gives these physical quantities short names. Part 1 will build the launch calculation from the picture before evaluating it.

In [5]:
# Dependencies
import subprocess, sys

# Only install packages that are not already importable in this environment
required = [("numpy", "numpy"), ("scipy", "scipy")]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

import numpy as np
from scipy import stats

np.random.seed(42)
print("All dependencies ready")

# Physical constants for the free kick scenario
g = 9.81          # gravity changes downward velocity by 9.81 m/s each second
v0 = 20.0        # launch speed (m/s)
BALL_RADIUS = 0.11
WALL_X = 9.15    # wall position (m)
WALL_H = 1.8     # wall height (m)
GOAL_X = 20.0    # front of the goal (m)
CROSS_H = 2.44   # crossbar height (m)
NET_X = 21.5     # back of the net (m)
TARGET_H = 1.10  # intended ball-centre height at back-net impact (m)
GOAL_BOTTOM = BALL_RADIUS
GOAL_TOP = CROSS_H - BALL_RADIUS


def ball_state(x, theta_deg):
    """Return time, height, and velocity when the ball reaches horizontal position x."""
    theta = np.radians(theta_deg)
    vx = v0 * np.cos(theta)
    t = x / vx
    vy = v0 * np.sin(theta) - g * t
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    return {"x": float(x), "t": float(t), "y": float(y), "vx": float(vx), "vy": float(vy)}


def ball_height(x, theta_deg):
    """Height of the ball centre at horizontal position x and launch angle theta (degrees)."""
    return ball_state(x, theta_deg)["y"]


print("\nFree kick setup:")
print(f"  Launch speed: {v0} m/s")
print(f"  Gravity changes downward velocity by {g} m/s each second")
print(f"  Wall at {WALL_X}m, ball centre must clear {WALL_H + BALL_RADIUS:.2f}m")
print(f"  Goal plane at {GOAL_X}m, legal centre window [{GOAL_BOTTOM:.2f}, {GOAL_TOP:.2f}]m")
print(f"  Back-net target at ({NET_X:.1f}m, {TARGET_H:.2f}m)")

  ok  numpy
  ok  scipy
All dependencies ready

Free kick setup:
  Launch speed: 20.0 m/s
  Gravity changes downward velocity by 9.81 m/s each second
  Wall at 9.15m, ball centre must clear 1.91m
  Goal plane at 20.0m, legal centre window [0.11, 2.33]m
  Back-net target at (21.5m, 1.10m)


---

## Part 1 — Dot Product as Useful Power Transfer

### 1. Feel the question

Freeze the instant the foot leaves the ball.

The ball has **one launch-speed arrow**. It points upward and forward at the same time.

Now imagine a thin rail from the ball to the target. Ask:

> **How much of the launch is actually pushing along that rail?**

A launch aimed far away from the rail wastes useful motion. A launch almost aligned with it transfers nearly all its speed toward the intended route.

The upward part is not useless. It buys height while gravity pulls downward. But it is still useful to separate:

- motion **along the target line**;
- motion **across the target line**.

![Dot product visualized as the signed shadow of the kick arrow along the direct target line](images/free-kick-vector-projection.svg)

The green shadow is the part of the launch arrow that lies along the target line. Longer shadow, stronger alignment.

### 2. Put friendly numbers on it

The target rises only `1.10 m` over a `21.5 m` run: about **5 cm up for every metre forward**. So the direct target line is shallow, about `2.93°`.

Gravity means the ball must start above that line. Try three launch angles at the same `20 m/s` speed:

| Launch angle | Height at the back-net target | Read it physically |
| ---: | ---: | --- |
| `15°` | `−0.31 m` | Too low; gravity wins too early. |
| `19.11°` | `1.10 m` | Reaches the chosen target. |
| `25°` | `3.13 m` | Too high. |

So the working launch is `19.11°`, while the direct target line is `2.93°`. The gap is about `16.18°`.

Scale both arrows to length `1`. The launch casts a shadow of about `0.960` onto the target line. Restore the real launch speed:

> **useful speed along the target line ≈ `20 × 0.960 = 19.2 m/s`**

Almost all the launch supports the intended route.

### 3. Name the math

Cosine turns an angle gap into a shadow fraction. If $\Delta\theta$ is the angle between the launch and target directions:

$$\text{alignment}=\cos(\Delta\theta)$$

For vectors, this useful-shadow calculation is the **dot product**:

$$\mathbf{v}\cdot\hat{\mathbf{t}}=\lVert\mathbf{v}\rVert\cos(\Delta\theta)$$

Here $\mathbf{v}$ is the launch velocity and $\hat{\mathbf{t}}$ is a unit arrow along the target line.

> **Dot product = how much of one arrow is useful in the direction of another.**

#### Predict First

A `16.18°` gap is fairly small. What should the useful fraction look like?

1. Close to the full launch
2. About half the launch
3. Negative, as if pointing backward

Make the physical prediction before running the code.

### Calibrate Your Intuition

Keep the launch arrow fixed. Rotate only the direction you care about.

- **Same direction:** the whole arrow is useful → score `+1`.
- **Sideways:** none of the arrow moves along that direction → score `0`.
- **Backward:** the arrow fights that direction → score `−1`.
- **This kick:** the directions differ by only `16.18°` → score about `+0.960`.

The sign tells you **with or against**. The size tells you **how much**.

For the real `20 m/s` launch, the code first compares unit arrows, so it prints `0.9604`. Multiplying by `20 m/s` converts that fraction back into about `19.2 m/s` of target-aligned speed.

<details>
<summary><strong>How two arrows become coordinate lists</strong></summary>

A unit arrow at `19.11°` is mostly horizontal:

> **launch direction ≈ `[0.945, 0.327]`**

The direct target line at `2.93°` is even more horizontal:

> **target direction ≈ `[0.999, 0.051]`**

The computer pairs matching directions:

- horizontal agreement: `0.945 × 0.999 ≈ 0.944`;
- vertical agreement: `0.327 × 0.051 ≈ 0.017`.

Add them:

$$0.944+0.017\approx0.960$$

Only after seeing that concrete calculation do we write the general dot-product rule:

$$\mathbf{a}\cdot\mathbf{b}=a_xb_x+a_yb_y$$

</details>

### Before You Run the Code

Read the program as three physical moves:

1. Find the launch angle that lands at the target height.
2. Turn the launch and target directions into unit arrows.
3. Measure the launch arrow's shadow on the target arrow.

Expected result: **positive and close to `1`**, because the arrows are nearly aligned.

In [10]:
# Part 1: solve the pictured geometry, then compare the two directions
from scipy.optimize import brentq

# The direct target angle comes from the rise-to-run ratio explained above.
target_line_deg = np.degrees(np.arctan2(TARGET_H, NET_X))

# Find the lower angle where predicted target height minus intended height is zero.
def target_height_error(angle_deg):
    return ball_height(NET_X, angle_deg) - TARGET_H


launch_angle_deg = brentq(target_height_error, 5.0, 45.0)
assert np.isclose(target_height_error(launch_angle_deg), 0.0)

kick_dir = np.array([
    np.cos(np.radians(launch_angle_deg)),
    np.sin(np.radians(launch_angle_deg)),
])
target_dir = np.array([
    np.cos(np.radians(target_line_deg)),
    np.sin(np.radians(target_line_deg)),
])

# For unit vectors, the dot product is the signed projection length.
shadow_score = np.dot(kick_dir, target_dir)
angle_between = np.degrees(np.arccos(np.clip(shadow_score, -1, 1)))

print(f"Kick direction ({launch_angle_deg:.2f}°):   {kick_dir.round(3)}")
print(f"Target direction ({target_line_deg:.2f}°): {target_dir.round(3)}")
print(f"Angle between arrows: {angle_between:.2f}°")
print(f"Forward-shadow score: {shadow_score:.4f} out of 1")
print()
print("Interpretation: most of the launch direction points along the direct target line.")

Kick direction (19.11°):   [0.945 0.327]
Target direction (2.93°): [0.999 0.051]
Angle between arrows: 16.18°
Forward-shadow score: 0.9604 out of 1

Interpretation: most of the launch direction points along the direct target line.


#### What the Number Confirms

The code prints **0.9604**.

That means about **96% of a unit launch arrow** lies along the direct target line. At `20 m/s`, that corresponds to about **19.2 m/s of useful target-aligned motion**.

The dot product answers a directional question at contact. It does **not** tell us how the arrow changes after launch.

For that, stop the movie half a second later and inspect the ball's velocity arrow.

<details>
<summary><strong>Try a different launch direction</strong></summary>

```python
# Try 90° for a sideways launch, then 200° for a backward-pointing launch.
kick_angle = 90
target_angle = target_line_deg

kick = np.array([np.cos(np.radians(kick_angle)), np.sin(np.radians(kick_angle))])
target = np.array([np.cos(np.radians(target_angle)), np.sin(np.radians(target_angle))])
score = np.dot(kick, target)
print(f"kick={kick_angle}°, target={target_angle:.2f}° → shadow score={score:.4f}")
```

</details>

---

## Part 2 — Derivative as a Freeze-Frame Speedometer

### 1. Stop the movie

Freeze the ball exactly **half a second after launch**.

It is near the wall, still moving forward, and still rising slightly.

Now perform a strange thought experiment:

> **What direction would the ball fly if gravity vanished at this exact microsecond?**

It would leave the curved path along a straight arrow touching the curve at that point. That touching line is the **tangent**.

The tangent is not the whole future. It is the ball's direction **right now**.

### 2. Read the frozen speedometer

At `t = 0.50 s`, the gravity-only model gives approximately:

- position: `(9.45 m, 2.05 m)`;
- horizontal velocity: `18.90 m/s`;
- vertical velocity: `+1.64 m/s`.

The arrow is mostly forward and only slightly upward. That matches the picture: the ball is close to the top of its arc.

Over the next `0.04 s`, the frozen velocity alone would carry it roughly:

- forward: `18.90 × 0.04 ≈ 0.76 m`;
- upward: `1.64 × 0.04 ≈ 0.066 m`.

Gravity trims about `0.008 m` from that upward move, so the actual rise is about `0.058 m`.

### 3. Name the math

A **derivative** is this freeze-frame speedometer: position change per unit time at one instant.

$$v_x=\frac{dx}{dt},\qquad v_y=\frac{dy}{dt}$$

The tangent's slope is simply rise-rate divided by run-rate:

$$\frac{dy}{dx}=\frac{v_y}{v_x}$$

At the peak, the freeze-frame speedometer reads $v_y=0$. The ball is still moving forward, but for one instant it is neither rising nor falling.

> **Derivative = the arrow the ball would follow if the forces switched off right now.**

### Turn One Frozen Frame into One Tiny Move

The derivative gives the velocity arrow **now**. To advance the movie, let that arrow act for one short time slice.

![One exact gravity-only update showing the horizontal move, vertical move, and gravity correction at step 10](images/gravity-only-step.svg)

### 1. Feel the update

Current velocity carries the ball. Gravity bends that motion slightly downward. The endpoint becomes the next frame.

### 2. Use the step-10 numbers

One slice lasts about `0.041 s`.

- Current vertical velocity would carry the ball up by about `0.120 m`.
- Gravity's downward effect ramps up during the slice and removes about `0.008 m`.
- Net rise: `0.120 − 0.008 = 0.112 m`.

Horizontally, no force acts in this model, so the ball moves forward about `0.768 m`.

### 3. Write the compact update

Distance is speed times a tiny duration:

$$\Delta x=v_x\Delta t$$

Vertical motion gets the same velocity contribution, minus gravity's small accumulated drop:

$$\Delta y=v_y\Delta t-\frac12g(\Delta t)^2$$

Gravity also changes the speedometer reading for the next frame:

$$v_{y,\text{next}}=v_y-g\Delta t$$

The next code cell repeats this one understandable move 28 times.

In [8]:
# Part 2: repeat the gravity-only update derived above
theta_deg = launch_angle_deg
theta_rad = np.radians(theta_deg)
step_count = 28

velocity_x = v0 * np.cos(theta_rad)
position_x = 0.0
position_y = 0.0
velocity_y = v0 * np.sin(theta_rad)
flight_time = NET_X / velocity_x
delta_t = flight_time / step_count

step_rows = []
for step_index in range(step_count):
    delta_x = velocity_x * delta_t
    delta_y = velocity_y * delta_t - 0.5 * g * delta_t**2
    next_velocity_y = velocity_y - g * delta_t

    if step_index in {0, 9, 18, 27}:
        step_rows.append((step_index + 1, position_x, position_y, delta_x, delta_y, next_velocity_y))

    position_x += delta_x
    position_y += delta_y
    velocity_y = next_velocity_y

# Compare the accumulated small steps with the direct projectile formula and target.
net_state = ball_state(NET_X, theta_deg)
step_error = np.hypot(position_x - net_state["x"], position_y - net_state["y"])
target_error = abs(net_state["y"] - TARGET_H)

print(f"28 slices across {flight_time:.3f}s give Δt={delta_t:.3f}s\n")
print(" step   start (x, y)       Δx        Δy       next vy")
for step_number, start_x, start_y, change_x, change_y, next_vy in step_rows:
    print(
        f" {step_number:>2d}    ({start_x:>5.2f}, {start_y:>4.2f})   "
        f"{change_x:+.3f}m   {change_y:+.3f}m   {next_vy:+.3f}m/s"
    )

print(f"\nAccumulated endpoint: ({position_x:.6f}, {position_y:.6f}) m")
print(f"Direct-formula endpoint: ({net_state['x']:.6f}, {net_state['y']:.6f}) m")
print(f"Intended target: ({NET_X:.6f}, {TARGET_H:.6f}) m")
print(f"Endpoint disagreement: {step_error:.2e} m")
print(f"Target-height error: {target_error:.2e} m")
assert step_error < 1e-10
assert target_error < 1e-10

x_peak_analytic = v0**2 * np.sin(2 * theta_rad) / (2 * g)
peak_state = ball_state(x_peak_analytic, theta_deg)
wall_state = ball_state(WALL_X, theta_deg)
height_left_after_wall = peak_state["y"] - wall_state["y"]
print(f"\nWall clearance: x={WALL_X:.2f}m, y={wall_state['y']:.2f}m, vy={wall_state['vy']:+.2f}m/s")
print(f"Turning point: x={x_peak_analytic:.2f}m, y={peak_state['y']:.2f}m, vy={peak_state['vy']:+.2e}m/s")
print(f"Height still to rise after the wall: {height_left_after_wall:.2f}m")
print("Prediction check: answer 2 — wall clearance happens near the top of the arc.")

28 slices across 1.138s give Δt=0.041s

 step   start (x, y)       Δx        Δy       next vy
  1    ( 0.00, 0.00)   +0.768m   +0.258m   +6.149m/s
 10    ( 6.91, 1.74)   +0.768m   +0.112m   +2.561m/s
 19    (13.82, 2.16)   +0.768m   -0.034m   -1.026m/s
 28    (20.73, 1.28)   +0.768m   -0.179m   -4.613m/s

Accumulated endpoint: (21.500000, 1.100000) m
Direct-formula endpoint: (21.500000, 1.100000) m
Intended target: (21.500000, 1.100000) m
Endpoint disagreement: 7.16e-15 m
Target-height error: 1.33e-15 m

Wall clearance: x=9.15m, y=2.02m, vy=+1.80m/s
Turning point: x=12.61m, y=2.18m, vy=-8.88e-16m/s
Height still to rise after the wall: 0.16m
Prediction check: answer 2 — wall clearance happens near the top of the arc.


#### What the Freeze Frames Reveal

At the wall, the ball's centre is about **2.02 m** high. Its vertical speed has fallen to only **+1.80 m/s**.

The ball rises just **0.16 m more** before the vertical speedometer reaches zero at the **2.18 m peak**.

Then the sign flips:

- $v_y>0$: rising;
- $v_y=0$: at the peak;
- $v_y<0$: falling.

One number tells the phase of the flight. That is the derivative doing useful work.

## Integration as Stitching Movie Frames

### 1. Watch the frames accumulate

One freeze-frame arrow gives one tiny move. It does not give a whole free kick.

So repeat:

> **read the speedometer → move a little → let gravity change the speedometer → freeze again**

![Animation showing 4, 8, 16, and 28 repeated local predictions becoming a progressively smoother free-kick curve](images/calculus-accumulation.gif)

With only `4` frames, the construction looks chunky. With `28`, the tiny moves hug the smooth arc.

### 2. Read a few stitched frames

The code prints four snapshots from the 28-step movie:

| Step | Horizontal move | Vertical move | What you see |
| ---: | ---: | ---: | --- |
| `1` | `+0.768 m` | `+0.258 m` | Strong rise just after launch |
| `10` | `+0.768 m` | `+0.112 m` | Still rising, but less |
| `19` | `+0.768 m` | `−0.034 m` | Just past the peak |
| `28` | `+0.768 m` | `−0.179 m` | Falling into the target |

Add every horizontal move and every vertical move. The stitched endpoint is `(21.5 m, 1.1 m)`.

### 3. Name the math

For tiny slices, each vertical move is approximately current vertical speed times slice duration:

$$\Delta y_k\approx v_y(t_k)\Delta t$$

Stitch the movie by adding all those moves:

$$y(T)-y(0)\approx\sum_k v_y(t_k)\Delta t$$

Make the slices arbitrarily thin and the sum becomes an **integral**:

$$y(T)-y(0)=\int_0^T v_y(t)\,dt$$

> **Integration = rebuilding the journey from many freeze-frame speedometer readings.**

### Derivative and Integration Are a Pair

The free kick uses two complementary views:

- **Derivative:** pause the movie and read the velocity arrow now.
- **Integration:** stitch the tiny moves from all those arrows into a trajectory.

Neither needs a blueprint of the full arc. Each frame needs only the current position, current velocity, gravity, and a short time slice.

> **Model boundary:** real football also includes air resistance, wind, topspin, dip, and curl. We ignore them here so gravity is the only force changing the velocity.

This local-update pattern appears throughout AI: recurrent models, simulators, optimizers, and dynamical systems all carry a state forward one step at a time.

---

## Part 3 — A Matrix as a State-Moving Machine

### 1. Hold one frame in your hand

At `t = 0.50 s`, keep only two vertical facts:

> **state now = `[height, vertical speed] = [2.05, 1.64]`**

Think of this as a tiny state card carried from one movie frame to the next.

The next frame needs two answers:

1. What is the new height?
2. What is the new vertical speed?

### 2. Move the card forward by `0.04 s`

First imagine gravity paused for this one tiny slice.

- Height keeps its old `2.05 m` and gains `1.64 × 0.04 ≈ 0.066 m`.
- Vertical speed simply carries forward as `1.64 m/s`.

So the no-gravity prediction is roughly:

> **`[2.12, 1.64]`**

Now apply gravity's correction for the slice:

- height correction: about `−0.008 m`;
- velocity correction: about `−0.39 m/s`.

The next state is therefore about:

> **state next = `[2.11, 1.25]`**

### 3. Name the machine

The two no-gravity update rules can be stacked into one 2×2 table:

$$
\begin{bmatrix}
y_{\text{next}}\\
v_{y,\text{next}}
\end{bmatrix}
=
\begin{bmatrix}
1 & \Delta t\\
0 & 1
\end{bmatrix}
\begin{bmatrix}
y_{\text{now}}\\
v_{y,\text{now}}
\end{bmatrix}
+
\begin{bmatrix}
-\frac12g(\Delta t)^2\\
-g\Delta t
\end{bmatrix}
$$

Read the rows as plain instructions:

- first row: keep the old height and add `vertical speed × time`;
- second row: carry the old vertical speed forward;
- final vector: apply gravity's correction.

The 2×2 table is a **state-transition matrix**. Running both rows against the current state is **matrix multiplication**.

> **Matrix multiplication = several coordinated update rules acting on the same state card at once.**

> **AI connection:** a recurrent model does the same kind of work. It carries a compact state forward, mixes the old values, adds an update, and hands the new state to the next step.

### From a Physics Matrix to an AI Layer

The physics matrix above is chosen from known motion rules.

The existing code cell below keeps its original AI-flavoured example: it takes a normalized `[angle, speed]` feature vector and sends it through two weighted rows.

The surface story changes, but the computational move is identical:

> **one state vector enters → every row reads it → one output vector leaves**

In physics, we design the matrix. In machine learning, training often learns the matrix weights and bias.

In [ ]:
# Part 3: stack two dot-product recipes into one matrix
features = np.array([0.4, 0.8])  # normalized [angle signal, speed signal]

# Each row is one dot-product recipe applied to the same input vector.
W = np.array([
    [2.0, 0.5],   # row 1 listens strongly to angle
    [-0.5, 1.5],  # row 2 listens strongly to speed
])
b = np.array([0.1, -0.2])

output = W @ features + b

print(f"Input vector [angle, speed]: {features}")
print("\nRun each row as one dot product:")
for row_index, (row, bias, result) in enumerate(zip(W, b, output), start=1):
    angle_part, speed_part = row * features
    print(
        f"  row {row_index}: angle {angle_part:+.2f}, speed {speed_part:+.2f}, "
        f"bias {bias:+.2f} → result {result:.2f}"
    )

print(f"\nStacked matrix result: {output.round(3)}")
print("One row gives one output; two rows give an output vector of length two.")

### Read the Unchanged Code as Two Simultaneous Questions

The input state is the normalized feature card:

> **`[angle signal, speed signal] = [0.4, 0.8]`**

Each row reads the same card but cares about it differently.

| Row | Angle contribution | Speed contribution | Bias | Output |
| --- | ---: | ---: | ---: | ---: |
| `1` | `2 × 0.4 = 0.8` | `0.5 × 0.8 = 0.4` | `+0.1` | **1.3** |
| `2` | `−0.5 × 0.4 = −0.2` | `1.5 × 0.8 = 1.2` | `−0.2` | **0.8** |

Read it physically:

- Row 1 listens strongly to angle.
- Row 2 listens strongly to speed.
- Both rows act in the same instant.
- Their two answers travel together as `[1.3, 0.8]`.

These outputs are deliberately uncalibrated AI features, not metres or metres per second. The cell exists to expose the same mechanism as the state-transition matrix:

> **rows are update recipes; the matrix runs all recipes together.**

### The Shared AI Equation

We have now seen the same shape twice:

- **Physics:** current state → fixed motion rules → next state.
- **AI layer:** current features → learned weighted rules → next features.

For the unchanged code cell, name the pieces:

- $\mathbf{x}$: input feature vector `[0.4, 0.8]`;
- $W$: two stacked row recipes;
- $\mathbf{b}$: one small offset per row;
- $\mathbf{y}$: output vector `[1.3, 0.8]`.

Only now write the familiar layer equation:

$$\mathbf{y}=W\mathbf{x}+\mathbf{b}$$

Read it conversationally:

> **Every row looks at the same state, produces one answer, and all answers leave together.**

That is why matrices are everywhere in AI. They update many connected quantities in one organized move.

<details>
<summary><strong>Try changing one column</strong></summary>

```python
W_test = np.array([
    [3.0, 0.1],  # row 1 listens mostly to angle
    [0.1, 3.0],  # row 2 listens mostly to speed
])

normal = np.array([0.4, 0.8])
more_angle = np.array([0.8, 0.8])

print("normal:    ", W_test @ normal)
print("more angle:", W_test @ more_angle)
print("change:    ", W_test @ more_angle - W_test @ normal)
```

Only the angle feature changes. The first output moves much more because its row listens strongly to the angle column.

</details>

The free kick used exact inputs. Real players vary from attempt to attempt. That leads to probability.

---

## Part 4 — One Kick Is Not a Cloud of Attempts

So far the same angle and speed always produce the same curve. That describes a perfectly repeatable kick.

A real player is not perfectly repeatable. Ask for 20° one hundred times and the actual kicks form a cloud around 20°:

- many land close to the intention;
- some are a little high or low;
- a few are farther away.

That cloud is a **probability distribution**.

The legal scoring angles span roughly **18.44° to 21.86°**. Compare two aims:

- **19.11°** sends one exact kick to the chosen back-net point;
- **20.15°** centres the whole cloud inside the legal window.

The visual question comes first:

> **Which centre leaves more of the cloud between the two legal boundaries?**

#### Predict First

Suppose the typical miss is about 3°. Which aim scores more often over 100 attempts?

1. 19.11°, because it is best for one exact kick
2. 20.15°, because it centres the whole cloud inside the legal window
3. They must be identical

### Let the Cloud Become Basic Probability Notation

Give the uncertain executed angle a name: $\Theta$.

The statement

> executed angle lands between 18.44° and 21.86°

becomes

$$18.44°<\Theta<21.86°$$

Placing $P(\cdot)$ around an event means “the probability of this event”:

$$P(18.44°<\Theta<21.86°)$$

This expression means:

> **the fraction of the kick cloud that falls inside the legal window**

For a cloud centred at 20.15° with a typical spread of 3°, the model estimates a fraction of about `0.431`: roughly 43 scores per 100 attempts.

A bell-shaped **Gaussian distribution** is used as a convenient model for the execution cloud. It is an assumption, not a physical law of football. Real kick errors need not form a perfect bell curve.

<details>
<summary><strong>Optional advanced connection: turn probability into a surprise score</strong></summary>

A likely event should have a small surprise penalty; an unlikely event should have a large one.

- chance `0.9` → small surprise;
- chance `0.1` → large surprise.

Why use a logarithm? Probabilities from several independent events multiply, while their logarithms add. Addition is easier to accumulate and optimize across many observations.

Name the chance $p$. The surprise score is:

$$\text{surprise}=-\log(p)$$

The minus sign makes small probabilities produce large positive penalties. When $p$ is the probability assigned to the observed outcome, this score is called **negative log-likelihood**.

</details>

> **Keep the objects separate:** one exact input produces one trajectory. Repeated imperfect inputs produce a cloud of trajectories. Probability describes the cloud rather than pretending every attempt is identical.

In [4]:
# Part 4: compare one point-target aim with a reliable repeated-attempt aim
# Keep angles that clear the wall and cross fully inside the goal frame
scoreable_angles = [
    angle for angle in np.linspace(5, 60, 20_000)
    if (
        ball_height(WALL_X, angle) > WALL_H + BALL_RADIUS
        and GOAL_BOTTOM < ball_height(GOAL_X, angle) < GOAL_TOP
    )
]

if not scoreable_angles:
    raise RuntimeError("No launch angle satisfies the physical constraints")

theta_lo = min(scoreable_angles)
theta_hi = max(scoreable_angles)
point_target_aim = launch_angle_deg
cluster_centered_aim = (theta_lo + theta_hi) / 2
sigma = 3.0


def probability_of_scoring(intended_angle, execution_sigma=sigma):
    distribution = stats.norm(loc=intended_angle, scale=execution_sigma)
    return distribution.cdf(theta_hi) - distribution.cdf(theta_lo)


probability_point_target = probability_of_scoring(point_target_aim)
probability_cluster_centered = probability_of_scoring(cluster_centered_aim)
nll_point_target = -np.log(probability_point_target)
nll_cluster_centered = -np.log(probability_cluster_centered)

print(f"Legal angle window: [{theta_lo:.2f}°, {theta_hi:.2f}°]")
print(f"Typical execution spread: {sigma:.1f}°\n")
print("ONE EXACT INPUT VERSUS A DISTRIBUTION")
print(
    f"  Aim at one chosen point: {point_target_aim:5.2f}°  "
    f"expected scores per 100={100 * probability_point_target:4.1f}"
)
print(
    f"  Center the whole cluster: {cluster_centered_aim:5.2f}°  "
    f"expected scores per 100={100 * probability_cluster_centered:4.1f}"
)
print(f"  Extra expected scores per 100: {100 * (probability_cluster_centered - probability_point_target):+.1f}")
print(f"  Surprise penalty: {nll_point_target:.3f} → {nll_cluster_centered:.3f}")
print("\nPrediction check: answer (b) — centering the cluster leaves more room for imperfect kicks.\n")

print("Consistency check at the cluster-centered aim:")
for spread in [1.0, 2.0, 3.0, 5.0, 8.0, 10.0]:
    probability = probability_of_scoring(cluster_centered_aim, spread)
    print(f"  typical spread={spread:4.1f}°: expected scores per 100={100 * probability:4.1f}")

Legal angle window: [18.44°, 21.86°]
Typical execution spread: 3.0°

ONE EXACT INPUT VERSUS A DISTRIBUTION
  Aim at one chosen point: 19.11°  expected scores per 100=40.9
  Center the whole cluster: 20.15°  expected scores per 100=43.1
  Extra expected scores per 100: +2.3
  Surprise penalty: 0.895 → 0.841

Prediction check: answer (b) — centering the cluster leaves more room for imperfect kicks.

Consistency check at the cluster-centered aim:
  typical spread= 1.0°: expected scores per 100=91.3
  typical spread= 2.0°: expected scores per 100=60.8
  typical spread= 3.0°: expected scores per 100=43.1
  typical spread= 5.0°: expected scores per 100=26.8
  typical spread= 8.0°: expected scores per 100=16.9
  typical spread=10.0°: expected scores per 100=13.6


### Imagine 100 repeated kicks

With a 3° typical spread, the two aims behave differently over many attempts:

| Strategy | Intended angle | Expected scores out of 100 | What it describes |
| --- | ---: | ---: | --- |
| Hit one back-net point precisely | 19.11° | About **41** | One exact gravity-only trajectory |
| Center varied attempts in the legal window | 20.15° | About **43** | Probability of any legal score |

The gain is only about two extra goals per 100 kicks here, but the principle is durable:

> **A single predicted value and a distribution of possible values answer different questions.**

Consistency matters even more than the one-degree change in aim:

| Typical spread around the aim | Rough scores out of 100 at the centered aim | What the cluster looks like |
| ---: | ---: | --- |
| 1° | **91** | Tight and consistent |
| 2° | **61** | Wider, but most attempts remain near the aim |
| 3° | **43** | Many attempts spill outside the narrow legal window |
| 5° | **27** | Broad scatter |
| 8° | **17** | Very broad scatter |

Moving the **aim** shifts the centre of the distribution. Improving **consistency** tightens it. Probability lets us reason about both without pretending every repeated attempt is identical.

#### What just happened

The two questions preferred different reference angles:

- **19.11°** sends one exact kick to the chosen back-net point;
- **20.15°** gives a spread of imperfect kicks the most room inside the legal interval.

Over 100 attempts with a 3° typical spread, that shift raises expected scores from about **41 to 43**. The numerical surprise penalty also falls from about **0.895 to 0.841**; smaller means a legal score is less surprising under the distribution.

This is the bridge to machine learning:

> **A deterministic prediction says what one input produces. A probabilistic model says how plausible many possible outcomes are.**

#### Your turn — cost of poor consistency

Before changing the number, predict what a wider cluster will do:

```python
typical_spread = 8.0  # change to 1.0 after predicting the effect
chance = probability_of_scoring(cluster_centered_aim, typical_spread)
print(f"typical spread={typical_spread}°: expected scores per 100={100 * chance:.1f}")
print(f"surprise penalty={-np.log(chance + 1e-12):.3f}")
```

This calculation does not claim every football error follows a perfect bell curve. It shows how an assumption about uncertainty becomes a testable probability.

<details>
<summary><strong>Connect the surprise penalty to classification</strong></summary>

The formal name for the surprise penalty is **negative log-likelihood**. In classification, cross-entropy is the negative log probability assigned to the correct class.

The Gaussian model here and the softmax probabilities used for classification are different distributions, but both use the same principle: **assign high probability to outcomes like the ones observed**.

</details>

---

## The Free-Kick Story in One Pass

- **Vectors** store direction and speed.
- **Dot products** measure useful alignment with the target.
- **Derivatives** read the velocity arrow in one frozen frame.
- **Integration** stitches tiny moves into the full flight.
- **Matrices** carry the ball's state from one frame to the next.
- **Probability** describes a cloud of imperfect repeated kicks.

One scenario carried every idea. The next notebook applies the same mathematical habits to fitted models.

→ **Next:** [`../01-ml-basics/ml-basics.ipynb`](../01-ml-basics/ml-basics.ipynb)